In [1]:
import numpy as np
import pandas as pd

In [2]:
import psycopg2

In [3]:
market = 'krakiddy'
#market = 'kraladies'

In [4]:
### don't forget to insert proper password

if market=='kraladies':
    conn = psycopg2.connect("dbname=kraladies user=sebastian host=localhost password=123456")
elif market=='krakiddy':    
    conn = psycopg2.connect("dbname=kkdj20_test user=sebastian host=localhost password=123456")
else:
    "something wrong. no market type selected"
    sys.exit(1)
cur = conn.cursor()
print (f"Will now insert into {market.upper()}!")

Will now insert into KRAKIDDY!


In [5]:
cur.execute("Select * from auth_user;")
cur.fetchall()

[(23,
  'pbkdf2_sha256$100000$MbHLiTijU4la$w7/b5GgHTE0NG/E9XfyftVJqsRz+lg+GSAwIZigR4z8=',
  datetime.datetime(2025, 4, 5, 8, 32, 34, 650890, tzinfo=datetime.timezone.utc),
  False,
  'desiree',
  'Desiree',
  'Schweizer',
  '',
  False,
  True,
  datetime.datetime(2022, 4, 19, 13, 24, 3, tzinfo=datetime.timezone.utc)),
 (26,
  'pbkdf2_sha256$100000$onV2wDcbYnLC$0AuS8LsG6BcXPM0SoarORziEhasRtNKrD3Q3vPOsU/g=',
  datetime.datetime(2025, 4, 5, 8, 35, 17, 769096, tzinfo=datetime.timezone.utc),
  False,
  'sarina',
  'Sarina',
  'Weber',
  '',
  True,
  True,
  datetime.datetime(2022, 4, 19, 13, 24, 3, 60336, tzinfo=datetime.timezone.utc)),
 (28,
  'pbkdf2_sha256$100000$t4PJCStQpvyk$1bxcDKyCP1DQ1HxfwV3S1oZn0JRmMm95KCC5Ch4k4v8=',
  datetime.datetime(2025, 4, 5, 8, 35, 54, 636685, tzinfo=datetime.timezone.utc),
  True,
  'pina',
  'Pina',
  'Chiovetta',
  '',
  True,
  True,
  datetime.datetime(2022, 4, 19, 13, 24, 3, 60336, tzinfo=datetime.timezone.utc)),
 (38,
  'pbkdf2_sha256$100000$CKoHnTbX

In [6]:
cur.execute("Select * from baskets_event;")
cur.fetchall()

[(1,
  'TestEvent',
  datetime.date(2022, 4, 19),
  'Only for testing',
  datetime.datetime(2022, 4, 19, 10, 7, 46, 110365, tzinfo=datetime.timezone.utc),
  1),
 (2,
  'KKM 23April',
  datetime.date(2022, 4, 23),
  '8. KRA-KIDDY-Markt 23. April 2022',
  datetime.datetime(2022, 4, 19, 10, 29, 46, 670461, tzinfo=datetime.timezone.utc),
  1),
 (3,
  'KKM9',
  datetime.date(2022, 9, 10),
  '9. KRA-KIDDY-Markt 10. September 2022',
  datetime.datetime(2022, 9, 7, 18, 58, 23, 399259, tzinfo=datetime.timezone.utc),
  1),
 (4,
  'KKM10',
  datetime.date(2023, 4, 22),
  '10. Kra-Kiddy-Markt am 22.04.2023 Jubilaeum!',
  datetime.datetime(2023, 4, 18, 19, 42, 49, 294441, tzinfo=datetime.timezone.utc),
  1),
 (5,
  'KKM11',
  datetime.date(2023, 9, 9),
  '11ter KraKiddy Markt',
  datetime.datetime(2023, 9, 8, 15, 38, 40, 868118, tzinfo=datetime.timezone.utc),
  1),
 (6,
  'KKM12',
  datetime.date(2024, 4, 20),
  '12ter Kra-Kiddy Markt',
  datetime.datetime(2024, 4, 17, 19, 18, 58, 947880, tzinfo=da

In [15]:
def add_vendor_to_event(vendor_id, event_id, dryrun=True, verbose=True):
    ## check if this realtion already exists:
    sql_cmd = "SELECT id FROM baskets_vendor_events WHERE vendor_id=%s AND event_id=%s"
    cur.execute(sql_cmd, (vendor_id, event_id))
    res = cur.fetchall()
    if not res:
        ## if not, add realtion to db:
        if verbose:
            print ("relation not found. create relation.")
        if not dryrun:
            sql_cmd = "INSERT INTO baskets_vendor_events (vendor_id, event_id) VALUES (%s, %s)"
            cur.execute(sql_cmd, (vendor_id, event_id))
    else:
        if verbose:
            print ('  vendor_id={} ALREADY assigned to event_id={}. Assignment_id={}'.format(vendor_id, event_id,res[0][0]))

In [12]:
def add_vendor_to_db(number, fname, lname, address=None, phone=None, dryrun=True, verbose=True):
    if verbose:
        print ("Adding {} {} to vendors...".format(fname,lname) )
    if not dryrun:
        sql_cmd = "INSERT INTO baskets_vendor (vendor_number, first_name, last_name, address, phone) VALUES (%s,%s,%s,%s,%s)"
        cur.execute(sql_cmd, (number, fname, lname, address, phone))
        cur.execute("SELECT id FROM baskets_vendor WHERE first_name=%s AND last_name=%s",(fname, lname))
        res = cur.fetchall()
        if not res:
            print ("Something went terribly wrong! Tryed to insert vendor {} {} to db, but not found back".format(fname,lname))
            sys.exit(1)
        else:
            if verbose:
                print ("  ... success. {} {} has vendor_id={}".format(fname,lname,res[0]))
    else:
        if verbose:
            print ("   (address={}, phone={}, number={})".format(address, phone, number))

In [8]:
from difflib import SequenceMatcher
def similar(a, b):
    return SequenceMatcher(None, a, b).ratio()    

In [14]:
def insert_vendor(number, name, address=None, phone=None, event_id=0, dryrun=True, verbose=True):
    try:
        fname,lname=name.split()
    except ValueError:
        if '.' in name:
            fname,lname=name.split('.')
            fname+='.'
        else:
            fname=name
            lname="TNr_{}".format(number)
        print ("ValueError for {}. Assigning (fname,lname)=({},{})".format(name, fname, lname))
    #print ("number={}, first_name={} last_name={}, address={}, phone={}".format(number,fname,lname,address,phone))
    cur.execute("SELECT id, vendor_number FROM baskets_vendor WHERE first_name=%s AND last_name=%s",(fname, lname))
    res = cur.fetchall()
    if not res:
        #print("Vendor {} {} not found. Pleas insert!".format(fname,lname))
        ## Uncomment the following in case it is needed:
        
        cur.execute("SELECT id, vendor_number, first_name, last_name FROM baskets_vendor WHERE vendor_number=%s", (number,))
        res = cur.fetchall()
        if res:
            assert len(res) == 1
            score = similar(res[0][2], fname) * similar(res[0][3],lname)
            print ('ATTENTION: vendor_number {} exists, but names do not agree! Maybe typo?'.format(number))
            print ('  Similar?: {}=={}? {}=={}?, score={:.3f}'.format(res[0][2], fname, res[0][3], lname, score))
            return -1
        
        add_vendor_to_db(number=number, fname=fname, lname=lname, address=address, phone=phone, dryrun=dryrun, verbose=verbose)
        ## NOW the vendor has to be in the database. Verify!
        cur.execute("SELECT id, vendor_number FROM baskets_vendor WHERE first_name=%s AND last_name=%s",(fname, lname))
        res = cur.fetchall()
    if not res:
        # print("Vendor {} {} not found. Pleas insert!".format(fname,lname))
        ## Uncomment the following in case it is needed:
        # add_vendor_to_db(number=number, fname=fname, lname=lname, address=address, phone=phone, dryrun=dryrun)
        if not dryrun:
            print("Error. Cannot insert vendor #{} ({},{}) into database. Break".format(number,fname,lname))
    else:
        assert len(res)==1
        vendor_id = res[0][0]
        vendor_number = res[0][1]
        if vendor_number == number:
            if verbose:
                print ("Vendor #{} ({} {}) has pk={}".format(vendor_number, fname, lname, vendor_id))
            add_vendor_to_event(vendor_id=vendor_id, event_id=event_id, dryrun=dryrun, verbose=verbose)
        else:
            print ("Vendor pk={} ({} {}) already exists but has #{} (instead of #{})!".format(vendor_id, fname, lname, vendor_number, number))
            print ("type(number)={}, type(vendor_number)={}".format(type(number),type(vendor_number)))

In [15]:
!head Teilnehmer_7Sep2024.csv

In [10]:
!head TN_29_11_2024_Kraladies1.csv

Teilnehmerliste KRA-LADIES Fashionmarkt 29.11.2024,,,,
11/29/2024,,,,
NR,Name,Vorname,TN-Umschlag erhalten,Unterschrift
2,Martens,Melissa,,
3,Müller,Janine,,
4,Blust,Laura,,
5,Blum,Tatjana,,
6,Tritschler,Tamara,,
8,Sorgente,Eugenie,,
9,Ehret,Sonja,,


In [2]:
!head Teilnehmer_05_04_2025.csv

Lfd.Nr,Teilnehmer-Nummer,Name,Anschrift/ Kontakt,TN 05.04.25,Unterschrift
1,1,KRA-KIDDY,KRäftig Anpacken für KIDS,,
2,4,Desiree Schweizer,,,
3,5,Manu Blum,manuela-blum2016@web.de,,
4,7,Manuela Zimmermann,zimmermann_manu@t-online.de,,
5,17,Tati Blum,Pfarrgarten/ 79369 Wyhl,,
6,19,Pina Lindemann,Stockacker Str/ Wyhl,,
7,22,Sandra Hiss,Ohnestalweg 5/ Kiechlinsbergen,,
8,25,Janine Müller,Ranzenstr.47/Forchheim; janinemue@web.de,,
9,26,Tini Schwörer,Endingerstr./ Wyhl,,


In [11]:
!head TN_20-09-2025.csv

Lfd.Nr,Teilnehmer-Nummer,Name,Anschrift/ Kontakt,TN 20.09.25,Unterschrift
1,1,KRA-KIDDY,KRäftig Anpacken für KIDS,,
2,4,Desiree Schweizer,,,
3,5,Manu Blum,,,
4,13,Janine Isele,,,
5,14,Julia Kollinger,,,
6,17,Tati Blum,,,
7,18,Nadja Steiger,,,
8,19,Pina Lindemann,,,
9,22,Sandra Hiss,,,


In [24]:
EVENT_ID = 1 #KraKiddy:  id1: TestEvent ; id2: 8. KRA-KIDDY-Markt 23. April 2022 ; id3: KKM9 (9. KRA-KIDDY-Markt am 10. September 2022)
             #  id9: KKM15 (20Sep2025)
             #KraLadies: id1: TestEvent ; id2: KL1
dryrun = True ## Attention False will actually do the insert!
verbose = True
TN_liste="TN_20-09-2025.csv"

line_nr = 0

skip_n_lines = 1
#with open("Teilnehmerliste_22_04_2023.csv") as infile:
with open(TN_liste) as infile:
 
    for _ in range(skip_n_lines):
        next(infile)
    for line in infile:
        line_nr+=1
        w = line.strip().split(",")
        ## use this for KraKiddy TN Files:
        if market=='krakiddy' and len(w) >= 4:
            print(24*"*"+"\n***** Line {} *****\n".format(line_nr)+24*"*")
            insert_vendor(number=int(w[1]),name=w[2],address=w[3], event_id=EVENT_ID, dryrun=dryrun, verbose=verbose)
        elif market=='kraladies' and len(w) >= 4:
            print(24*"*"+"\n***** Line {} *****\n".format(line_nr)+24*"*")
            insert_vendor(number=int(w[0]),name=w[2]+" "+w[1], address=w[3], event_id=EVENT_ID, dryrun=dryrun, verbose=verbose)
        else:
            print (len(w))
            print ("Fatal error!")
            print ("\t",w)
            import sys
            sys.exit(0)

************************
***** Line 1 *****
************************
ValueError for KRA-KIDDY. Assigning (fname,lname)=(KRA-KIDDY,TNr_1)
ATTENTION: vendor_number 1 exists, but names do not agree! Maybe typo?
  Similar?: KRA-KIDDY==KRA-KIDDY? e.V.==TNr_1?, score=0.000
************************
***** Line 2 *****
************************
Vendor #4 (Desiree Schweizer) has pk=2
  vendor_id=2 ALREADY assigned to event_id=1. Assignment_id=71
************************
***** Line 3 *****
************************
Vendor #5 (Manu Blum) has pk=3
  vendor_id=3 ALREADY assigned to event_id=1. Assignment_id=72
************************
***** Line 4 *****
************************
Vendor #13 (Janine Isele) has pk=96
  vendor_id=96 ALREADY assigned to event_id=1. Assignment_id=707
************************
***** Line 5 *****
************************
Vendor #14 (Julia Kollinger) has pk=5
  vendor_id=5 ALREADY assigned to event_id=1. Assignment_id=74
************************
***** Line 6 *****
**************

In [23]:
conn.commit()

In [26]:
_eventID=9
cur.execute('''SELECT bv.vendor_number AS vn
                FROM baskets_vendor_events AS bve
                JOIN baskets_vendor AS bv 
                    ON bv.id=bve.vendor_id
                WHERE event_id=%s 
                GROUP BY vn
                ORDER BY vn
            ''',(_eventID,))
li = [vid[0] for vid in cur.fetchall()]
print (f'{len(li)} Teilnehmer in event_id={_eventID}. \nTeilnehmer-Nummern: ')
print (li)

80 Teilnehmer in event_id=9. 
Teilnehmer-Nummern: 
[1, 4, 5, 13, 14, 17, 18, 19, 22, 26, 30, 33, 34, 38, 44, 45, 47, 49, 55, 57, 58, 59, 63, 73, 85, 111, 115, 116, 120, 138, 140, 157, 161, 167, 170, 185, 188, 193, 195, 202, 204, 206, 209, 210, 212, 213, 214, 216, 218, 223, 227, 228, 230, 231, 235, 245, 250, 251, 254, 316, 319, 403, 404, 414, 416, 418, 419, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 435, 444]


In [23]:
!grep -i '210' Kra_Kiddy_Markt_10Sep2022_Verkaeufer_20220904.csv

In [18]:
!grep -i '227' TN_Liste_20_04_2024.csv

72,227,Paulina Weis,b.paulaa@interia.pl,,,
